In [ ]:
# ============================================================
# PAPER 02 - HRFEST 2026
# ETAPA 1
# LIMPIEZA DEL CSV Y SEPARACION DEFINITIVA G1 / G2
#
# G1 = adquisicion real aproximadamente cada 1 minuto
# G2 = adquisicion real aproximadamente cada 10 minutos
#
# IMPORTANTE:
# - No se interpola temperatura.
# - No se hace resampling.
# - No se eliminan gaps internos.
# - Se eliminan timestamps invalidos y fechas fuera de 2026.
# - Se conservan las observaciones reales.
# ============================================================

import pandas as pd
import numpy as np
import csv
from google.colab import files


# ============================================================
# 1. SUBIR CSV ORIGINAL
# ============================================================

uploaded = files.upload()

filename = list(uploaded.keys())[0]

print("\nArchivo cargado:")
print(filename)


# ============================================================
# 2. LEER CSV DE FORMA ROBUSTA
# ============================================================
#
# El CSV original presenta filas con mas campos que los
# declarados en el encabezado.
#
# Para mantener consistencia con la estructura MongoDB
# exportada, se conservan exclusivamente las columnas
# declaradas en el encabezado original.
# ============================================================

rows = []

with open(
    filename,
    "r",
    encoding="utf-8-sig",
    errors="replace"
) as f:

    reader = csv.reader(f)

    header = next(reader)

    expected_cols = len(header)

    total_rows = 0
    rows_extra = 0
    rows_extra_nonempty = 0
    rows_missing = 0

    for row in reader:

        total_rows += 1

        # --------------------------------------------
        # Filas con columnas adicionales
        # --------------------------------------------

        if len(row) > expected_cols:

            rows_extra += 1

            extra = row[expected_cols:]

            if any(
                str(x).strip() != ""
                for x in extra
            ):
                rows_extra_nonempty += 1

            row = row[:expected_cols]

        # --------------------------------------------
        # Filas con columnas faltantes
        # --------------------------------------------

        elif len(row) < expected_cols:

            rows_missing += 1

            row = row + (
                [""] *
                (expected_cols - len(row))
            )

        rows.append(row)


df = pd.DataFrame(
    rows,
    columns=header
)


print("\n====================================================")
print("CSV STRUCTURE REPORT")
print("====================================================")

print("Header:")
print(header)

print("\nExpected columns:", expected_cols)

print("Total data rows:", total_rows)

print(
    "Rows with extra columns:",
    rows_extra
)

print(
    "Rows with non-empty extra columns:",
    rows_extra_nonempty
)

print(
    "Rows with missing columns:",
    rows_missing
)

print(
    "Rows preserved:",
    len(df)
)


# ============================================================
# 3. COLUMNAS PRINCIPALES
# ============================================================

datetime_col = "datetime"
temperature_col = "dsTemperature"

print("\n====================================================")
print("AVAILABLE COLUMNS")
print("====================================================")

for col in df.columns:
    print("-", col)


if datetime_col not in df.columns:

    raise ValueError(
        f"No se encontro la columna '{datetime_col}'."
    )


if temperature_col not in df.columns:

    raise ValueError(
        f"No se encontro la columna '{temperature_col}'."
    )


# ============================================================
# 4. CONVERSION DE TIPOS
# ============================================================

df["datetime_original"] = df[datetime_col]


df[datetime_col] = pd.to_datetime(
    df[datetime_col],
    errors="coerce",
    utc=True
)


df[temperature_col] = pd.to_numeric(
    df[temperature_col],
    errors="coerce"
)


# ============================================================
# 5. VALIDACION DE DATETIME Y TEMPERATURA
# ============================================================

invalid_datetime = (
    df[datetime_col]
    .isna()
    .sum()
)

invalid_temperature = (
    df[temperature_col]
    .isna()
    .sum()
)


print("\n====================================================")
print("VALIDATION REPORT")
print("====================================================")

print(
    "Invalid datetime:",
    invalid_datetime
)

print(
    "Invalid temperature:",
    invalid_temperature
)


# ============================================================
# 6. ELIMINAR REGISTROS SIN DATETIME O TEMPERATURA
# ============================================================

df_clean = (
    df
    .dropna(
        subset=[
            datetime_col,
            temperature_col
        ]
    )
    .copy()
)


print(
    "Rows after datetime/temperature validation:",
    len(df_clean)
)


# ============================================================
# 7. VALIDAR AÑO DEL EXPERIMENTO
# ============================================================
#
# El experimento hospitalario corresponde a 2026.
#
# El CSV contiene timestamps anómalos en 2036.
# Estos registros se excluyen del análisis temporal.
# ============================================================

year_counts = (
    df_clean[datetime_col]
    .dt.year
    .value_counts()
    .sort_index()
)


print("\n====================================================")
print("YEARS DETECTED")
print("====================================================")

print(year_counts)


invalid_year_mask = (
    df_clean[datetime_col].dt.year != 2026
)


invalid_year_count = (
    invalid_year_mask.sum()
)


print(
    "\nRecords outside 2026:",
    invalid_year_count
)


if invalid_year_count > 0:

    print(
        "\nExamples of records outside 2026:"
    )

    print(
        df_clean.loc[
            invalid_year_mask,
            [
                datetime_col,
                temperature_col
            ]
        ]
        .head(20)
        .to_string(index=False)
    )


df_clean = (
    df_clean[
        df_clean[datetime_col].dt.year == 2026
    ]
    .copy()
)


# ============================================================
# 8. ORDEN CRONOLOGICO
# ============================================================

df_clean = (
    df_clean
    .sort_values(datetime_col)
    .reset_index(drop=True)
)


# ============================================================
# 9. TIMESTAMPS DUPLICADOS
# ============================================================

duplicate_mask = (
    df_clean
    .duplicated(
        subset=[datetime_col],
        keep="first"
    )
)


duplicate_count = (
    duplicate_mask.sum()
)


print("\n====================================================")
print("DUPLICATE TIMESTAMPS")
print("====================================================")

print(
    "Duplicate timestamps:",
    duplicate_count
)


df_clean = (
    df_clean
    .loc[~duplicate_mask]
    .reset_index(drop=True)
)


# ============================================================
# 10. CALCULAR INTERVALOS TEMPORALES
# ============================================================

df_clean["interval_min"] = (
    df_clean[datetime_col]
    .diff()
    .dt.total_seconds()
    / 60
)


print("\n====================================================")
print("GLOBAL INTERVAL DISTRIBUTION")
print("====================================================")

print(
    df_clean["interval_min"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


# ============================================================
# 11. DEFINIR CORTE REAL G1 / G2
# ============================================================
#
# La inspeccion empirica de los timestamps mostro:
#
# 2026-09-08 21:39:01 -> 1 min
# 2026-09-08 21:40:00 -> 0.983 min
# 2026-09-08 21:50:11 -> 10.183 min
# 2026-09-08 22:00:01 -> 9.833 min
# 2026-09-08 22:10:01 -> 10.000 min
#
# Por tanto:
#
# G1 termina en:
# 2026-09-08 21:40:00 UTC
#
# G2 comienza en:
# 2026-09-08 21:50:11 UTC
#
# ============================================================

G2_START = pd.Timestamp(
    "2026-09-08 21:50:11",
    tz="UTC"
)


G1 = (
    df_clean[
        df_clean[datetime_col] < G2_START
    ]
    .copy()
    .reset_index(drop=True)
)


G2 = (
    df_clean[
        df_clean[datetime_col] >= G2_START
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# 12. RECALCULAR INTERVALOS DENTRO DE CADA GRUPO
# ============================================================

G1["interval_min"] = (
    G1[datetime_col]
    .diff()
    .dt.total_seconds()
    / 60
)


G2["interval_min"] = (
    G2[datetime_col]
    .diff()
    .dt.total_seconds()
    / 60
)


# ============================================================
# 13. MOSTRAR TRANSICION
# ============================================================

print("\n====================================================")
print("G1 / G2 TRANSITION")
print("====================================================")


transition_preview = (
    df_clean[
        (
            df_clean[datetime_col] >=
            pd.Timestamp(
                "2026-09-08 21:30:00",
                tz="UTC"
            )
        )
        &
        (
            df_clean[datetime_col] <=
            pd.Timestamp(
                "2026-09-08 22:30:00",
                tz="UTC"
            )
        )
    ]
    [
        [
            datetime_col,
            temperature_col,
            "interval_min"
        ]
    ]
)


print(
    transition_preview
    .to_string(index=False)
)


# ============================================================
# 14. FUNCION DE RESUMEN
# ============================================================

def summarize_group(
    group,
    name,
    expected_interval
):

    print("\n====================================================")
    print(name)
    print("====================================================")

    N = len(group)

    start = group[datetime_col].min()
    end = group[datetime_col].max()

    duration_hours = (
        end - start
    ).total_seconds() / 3600

    duration_days = (
        duration_hours / 24
    )

    intervals = (
        group[datetime_col]
        .diff()
        .dt.total_seconds()
        / 60
    )

    print(
        "N observations:",
        N
    )

    print(
        "Start:",
        start
    )

    print(
        "End:",
        end
    )

    print(
        "Duration hours:",
        round(duration_hours, 4)
    )

    print(
        "Duration days:",
        round(duration_days, 4)
    )

    print(
        "Expected nominal interval:",
        expected_interval,
        "min"
    )

    print(
        "Median interval:",
        round(
            intervals.median(),
            4
        ),
        "min"
    )

    print(
        "Mean interval:",
        round(
            intervals.mean(),
            4
        ),
        "min"
    )

    print(
        "Minimum interval:",
        round(
            intervals.min(),
            4
        ),
        "min"
    )

    print(
        "Maximum interval:",
        round(
            intervals.max(),
            4
        ),
        "min"
    )

    print(
        "Temperature mean:",
        round(
            group[temperature_col].mean(),
            4
        ),
        "°C"
    )

    print(
        "Temperature median:",
        round(
            group[temperature_col].median(),
            4
        ),
        "°C"
    )

    print(
        "Temperature SD:",
        round(
            group[temperature_col].std(),
            4
        ),
        "°C"
    )

    print(
        "Temperature min:",
        group[temperature_col].min(),
        "°C"
    )

    print(
        "Temperature max:",
        group[temperature_col].max(),
        "°C"
    )


summarize_group(
    G1,
    "G1 - REAL ~1-MIN ACQUISITION",
    expected_interval=1
)


summarize_group(
    G2,
    "G2 - REAL ~10-MIN ACQUISITION",
    expected_interval=10
)


# ============================================================
# 15. GAP ANALYSIS
# ============================================================
#
# G1:
# gap > 1.5 min
#
# G2:
# gap > 15 min
#
# No se eliminan.
# Solo se identifican y cuantifican.
# ============================================================

def analyze_gaps(
    group,
    expected_interval,
    name
):

    temp = group.copy()

    temp["gap_min"] = (
        temp[datetime_col]
        .diff()
        .dt.total_seconds()
        / 60
    )

    threshold = (
        expected_interval * 1.5
    )

    gaps = (
        temp[
            temp["gap_min"] > threshold
        ]
        .copy()
    )


    print("\n====================================================")
    print("GAP ANALYSIS:", name)
    print("====================================================")

    print(
        "Nominal interval:",
        expected_interval,
        "min"
    )

    print(
        "Gap threshold:",
        threshold,
        "min"
    )

    print(
        "Number of gaps:",
        len(gaps)
    )


    if len(gaps) > 0:

        print(
            "Maximum gap:",
            round(
                gaps["gap_min"].max(),
                4
            ),
            "min"
        )

        print(
            "Median gap:",
            round(
                gaps["gap_min"].median(),
                4
            ),
            "min"
        )

        print(
            "Mean gap:",
            round(
                gaps["gap_min"].mean(),
                4
            ),
            "min"
        )


        print("\nLargest gaps:")

        print(
            gaps[
                [
                    datetime_col,
                    "gap_min"
                ]
            ]
            .sort_values(
                "gap_min",
                ascending=False
            )
            .head(20)
            .to_string(index=False)
        )


    # --------------------------------------------------------
    # ESTIMATED MISSING OBSERVATIONS
    # --------------------------------------------------------
    #
    # Para cada gap:
    #
    # missing ≈ round(gap / nominal_interval) - 1
    #
    # Nunca puede ser negativo.
    # --------------------------------------------------------

    if len(gaps) > 0:

        gaps["estimated_missing"] = (
            np.maximum(
                np.round(
                    gaps["gap_min"]
                    / expected_interval
                ).astype(int) - 1,
                0
            )
        )

        estimated_missing = (
            gaps["estimated_missing"]
            .sum()
        )

    else:

        estimated_missing = 0


    expected_total = (
        len(group)
        + estimated_missing
    )


    if expected_total > 0:

        missing_percentage = (
            100
            * estimated_missing
            / expected_total
        )

    else:

        missing_percentage = np.nan


    print(
        "\nEstimated missing observations:",
        estimated_missing
    )

    print(
        "Estimated expected observations:",
        expected_total
    )

    print(
        "Estimated missing percentage:",
        round(
            missing_percentage,
            4
        ),
        "%"
    )


    return (
        gaps,
        estimated_missing,
        missing_percentage
    )


gaps_G1, missing_G1, missing_pct_G1 = (
    analyze_gaps(
        G1,
        expected_interval=1,
        name="G1"
    )
)


gaps_G2, missing_G2, missing_pct_G2 = (
    analyze_gaps(
        G2,
        expected_interval=10,
        name="G2"
    )
)


# ============================================================
# 16. VERIFICACION DEL REGIMEN NOMINAL
# ============================================================

def interval_regime_check(
    group,
    lower,
    upper,
    name
):

    intervals = (
        group[datetime_col]
        .diff()
        .dt.total_seconds()
        / 60
    )

    valid = intervals.dropna()

    inside = (
        (valid >= lower)
        &
        (valid <= upper)
    )

    percentage = (
        inside.mean() * 100
    )


    print("\n====================================================")
    print("INTERVAL REGIME CHECK:", name)
    print("====================================================")

    print(
        f"Intervals between {lower} and {upper} min:",
        int(inside.sum())
    )

    print(
        "Total intervals:",
        len(valid)
    )

    print(
        "Percentage:",
        round(
            percentage,
            4
        ),
        "%"
    )


    return percentage


G1_regime_percentage = (
    interval_regime_check(
        G1,
        lower=0.5,
        upper=1.5,
        name="G1 ~1 min"
    )
)


G2_regime_percentage = (
    interval_regime_check(
        G2,
        lower=8.0,
        upper=12.0,
        name="G2 ~10 min"
    )
)


# ============================================================
# 17. CREAR TABLA RESUMEN G1 / G2
# ============================================================

def group_summary_row(
    group,
    name,
    nominal_interval,
    missing,
    missing_pct,
    regime_pct
):

    start = group[datetime_col].min()
    end = group[datetime_col].max()

    duration_hours = (
        end - start
    ).total_seconds() / 3600

    intervals = (
        group[datetime_col]
        .diff()
        .dt.total_seconds()
        / 60
    )


    return {
        "Group": name,

        "N": len(group),

        "Start": start,

        "End": end,

        "Duration_h":
            duration_hours,

        "Duration_days":
            duration_hours / 24,

        "Nominal_interval_min":
            nominal_interval,

        "Median_interval_min":
            intervals.median(),

        "Mean_interval_min":
            intervals.mean(),

        "Estimated_missing":
            missing,

        "Estimated_missing_pct":
            missing_pct,

        "Nominal_regime_pct":
            regime_pct,

        "Temperature_mean_C":
            group[temperature_col].mean(),

        "Temperature_median_C":
            group[temperature_col].median(),

        "Temperature_SD_C":
            group[temperature_col].std(),

        "Temperature_min_C":
            group[temperature_col].min(),

        "Temperature_max_C":
            group[temperature_col].max()
    }


summary = pd.DataFrame(
    [
        group_summary_row(
            G1,
            "G1_1min",
            1,
            missing_G1,
            missing_pct_G1,
            G1_regime_percentage
        ),

        group_summary_row(
            G2,
            "G2_10min",
            10,
            missing_G2,
            missing_pct_G2,
            G2_regime_percentage
        )
    ]
)


print("\n====================================================")
print("FINAL G1 / G2 SUMMARY")
print("====================================================")

print(
    summary
    .to_string(index=False)
)


# ============================================================
# 18. PREPARAR CSV DE EXPORTACION
# ============================================================
#
# Conservamos interval_min porque nos sera util posteriormente
# para analizar gaps.
#
# Eliminamos solamente datetime_original.
# ============================================================

G1_export = (
    G1
    .drop(
        columns=[
            "datetime_original"
        ],
        errors="ignore"
    )
    .copy()
)


G2_export = (
    G2
    .drop(
        columns=[
            "datetime_original"
        ],
        errors="ignore"
    )
    .copy()
)


# ============================================================
# 19. EXPORTAR ARCHIVOS
# ============================================================

G1_filename = (
    "plasma_G1_1min_FINAL.csv"
)

G2_filename = (
    "plasma_G2_10min_FINAL.csv"
)

summary_filename = (
    "G1_G2_summary.csv"
)

gaps_G1_filename = (
    "gaps_G1_FINAL.csv"
)

gaps_G2_filename = (
    "gaps_G2_FINAL.csv"
)

transition_filename = (
    "transition_G1_G2_FINAL.csv"
)


G1_export.to_csv(
    G1_filename,
    index=False
)


G2_export.to_csv(
    G2_filename,
    index=False
)


summary.to_csv(
    summary_filename,
    index=False
)


gaps_G1.to_csv(
    gaps_G1_filename,
    index=False
)


gaps_G2.to_csv(
    gaps_G2_filename,
    index=False
)


transition_preview.to_csv(
    transition_filename,
    index=False
)


print("\n====================================================")
print("FILES GENERATED")
print("====================================================")

print(G1_filename)
print(G2_filename)
print(summary_filename)
print(gaps_G1_filename)
print(gaps_G2_filename)
print(transition_filename)


# ============================================================
# 20. DESCARGAR ARCHIVOS
# ============================================================

files.download(
    G1_filename
)

files.download(
    G2_filename
)

files.download(
    summary_filename
)

files.download(
    gaps_G1_filename
)

files.download(
    gaps_G2_filename
)

files.download(
    transition_filename
)


print("\n====================================================")
print("PROCESS COMPLETED")
print("====================================================")

print(
    "No temperature interpolation or resampling "
    "was performed."
)

print(
    "G1 and G2 contain the original valid observations "
    "after timestamp/year validation."
)

Saving bio_data_10mindata_ult26_250926.csv to bio_data_10mindata_ult26_250926.csv

Archivo cargado:
bio_data_10mindata_ult26_250926.csv

CSV STRUCTURE REPORT
Header:
['_id', '__v', 'createdAt', 'datetime', 'doorStatus', 'dsTemperature', 'humidity', 'time_slot', 'username']

Expected columns: 9
Total data rows: 27778
Rows with extra columns: 26778
Rows with non-empty extra columns: 26725
Rows with missing columns: 0
Rows preserved: 27778

AVAILABLE COLUMNS
- _id
- __v
- createdAt
- datetime
- doorStatus
- dsTemperature
- humidity
- time_slot
- username

VALIDATION REPORT
Invalid datetime: 0
Invalid temperature: 2
Rows after datetime/temperature validation: 27776

YEARS DETECTED
datetime
2026    27763
2036       13
Name: count, dtype: int64

Records outside 2026: 13

Examples of records outside 2026:
                 datetime  dsTemperature
2036-02-07 06:28:17+00:00          -31.8
2036-02-07 06:30:00+00:00          -31.8
2036-02-07 06:40:00+00:00          -31.3
2036-02-07 06:50:00+00:00 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (4 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


PROCESS COMPLETED
No temperature interpolation or resampling was performed.
G1 and G2 contain the original valid observations after timestamp/year validation.
